# 프로젝트 개요
듀오링고 신규 사용자는 첫 레슨의 어느 단계에서 이탈하며, 어떤 사용자 행동이 첫 레슨 완료와 두 번째 레슨 시작에 관련되어 있는가?

1. 첫 레슨 이탈: 첫 레슨을 시작한 사용자가 완료하지 못하는 이유 탐색
2. 후속 학습 미진행: 첫 레슨을 완료했지만 두 번째 레슨을 시작하지 않는 이유 탐색

# 퍼널 설계
| 단계 | 이벤트 | 분석 목적 |
|---|---|---|
| 1. 첫 레슨 시작 | `lesson_started` | 분석 대상 사용자 확정 |
| 2. 첫 답안 제출 | `answer_submitted` | 학습 시작 직후 이탈 확인 |
| 3. 진행률 50% 도달 | `lesson_halfway_reached` | 레슨 중반까지의 유지 여부 |
| 4. 첫 레슨 완료 | `lesson_completed` | 첫 레슨 완료율 측정 |
| 5. 두 번째 레슨 시작 | `lesson_started` | 후속 학습 전환율 측정 |

- 대상: 대한민국·한국어 UI에서 영어 학습을 시작한 신규 사용자로 가정
- 신규 사용자 유입 기간: 2026년 7월 1일~9월 30일
- 이벤트 관측 종료일: 2026년 10월 9일
- 첫 레슨 완료 인정: 시작 후 24시간 이내
- 후속 학습 전환 인정: 첫 레슨 완료 후 7일 이내 두 번째 레슨 시작

# 검증 가설 목록
| 가설 | 내용 | 분석 지표 |
|---|---|---|
| H1 | 구독 안내 화면에 노출된 사용자는 두 번째 레슨 시작률이 낮을 것이다. | 구독 안내 노출 여부별 후속 전환율 |
| H2 | 말하기 문제를 건너뛴 사용자는 첫 레슨 완료율이 낮을 것이다. | 말하기 건너뛰기 여부별 완료율 |
| H3 | 첫 레슨 완료 시 에너지가 부족한 사용자는 두 번째 레슨 시작률이 낮을 것이다. | 에너지 잔량별 후속 전환율 |
| H4 | 힌트를 사용한 사용자는 첫 레슨 완료율이 높을 것이다. | 힌트 사용 여부별 완료율 |
| H5 | 일일 학습 목표 시간이 긴 사용자는 두 번째 레슨 시작률이 높을 것이다. | 일일 목표 시간별 후속 전환율 |

- H1·H3·H5: 첫 레슨 완료자만 대상으로 분석
- H2·H4: 첫 레슨 시작자를 대상으로 분석
- H4: 힌트를 사용할 기회가 있었던 오답 경험자도 별도로 비교
- 가설별 집단의 전환율 비교 및 필요시 카이제곱 검정
- 가설과 일치하지 않는 결과도 그대로 기록

---

# 데이터 간단 분석

듀오링고 신규 사용자 10,000명의 가상 데이터를 분석하기 전, CSV 구조와 기초 분포 확인.

- 이번 단계: 데이터 크기·컬럼·결측치, 사용자 특성, 이벤트 종류, 날짜 분포, 중복 확인
- 다음 단계: 5단계 퍼널 집계와 H1~H5 가설 검증

> 실제 듀오링고 사용자가 아닌 프로젝트용 가상 데이터.

## 1. 라이브러리 및 데이터 호출

In [1]:

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
# 1. 프로젝트 최상위 경로: 내 PC의 GitHub 저장소 위치
# 폴더 위치가 다를경우 아래 ROOT 문자열만 수정
ROOT = Path(r"C:\Users\joon5\Documents\github\duolingo-funnel-analysis")

# 2. 생성 데이터와 전처리 데이터의 폴더 분리
DATA_RAW_DIR = ROOT / "data" / "raw"
DATA_PROCESSED_DIR = ROOT / "data" / "processed"

# 3. 생성할 CSV 이름
USER_PROFILE_FILE = "users.csv"
EVENT_LOG_FILE = "event_logs.csv"

# 4. 실제 저장할 전체 파일 경로
USER_PROFILE_PATH = DATA_RAW_DIR / USER_PROFILE_FILE
EVENT_LOG_PATH = DATA_RAW_DIR / EVENT_LOG_FILE

# 다른 폴더에 잘못 저장되는 상황 방지용 프로젝트 경로 확인
if not ROOT.is_dir():
    raise FileNotFoundError("프로젝트 폴더가 없습니다. 위 ROOT 경로를 본인 PC에 맞게 수정하세요.")

# 저장 폴더가 없는 경우 생성, 이미 존재하면 유지
DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("프로젝트 폴더 :", ROOT)
print("사용자 저장 경로:", USER_PROFILE_PATH)
print("이벤트 저장 경로:", EVENT_LOG_PATH)
print("전처리 폴더   :", DATA_PROCESSED_DIR)

프로젝트 폴더 : C:\Users\joon5\Documents\github\duolingo-funnel-analysis
사용자 저장 경로: C:\Users\joon5\Documents\github\duolingo-funnel-analysis\data\raw\users.csv
이벤트 저장 경로: C:\Users\joon5\Documents\github\duolingo-funnel-analysis\data\raw\event_logs.csv
전처리 폴더   : C:\Users\joon5\Documents\github\duolingo-funnel-analysis\data\processed


In [3]:
# 원본 데이터 읽기
df_raw01 = pd.read_csv(USER_PROFILE_PATH)   # 사용자 프로필 데이터
df_raw02 = pd.read_csv(EVENT_LOG_PATH)      # 이벤트 로그 데이터

# 2. 데이터 전처리

| 순서 | 전처리 항목 | 작업 내용 |
|---|---|---|
| 1 | 원본 데이터 복사 | 원본 CSV를 유지하면서 분석용 데이터 생성 |
| 2 | 날짜형 변환 | `first_seen_at`, `event_time`을 datetime으로 변환 |
| 3 | JSON 속성 분리 | `event_properties` 내부 값을 개별 컬럼으로 추출 |
| 4 | 자료형 정리 | 정답 여부를 1/0으로 변환하고 숫자형 속성 정리 |
| 5 | 정렬 | 사용자별 이벤트 시간순 정렬 |


In [4]:
import json

# 원본 DataFrame은 그대로 두고 전처리용 복사본 생성
df_users = df_raw01.copy()
df_events = df_raw02.copy()

# 날짜 문자열을 날짜 자료형으로 변환
df_users["first_seen_at"] = pd.to_datetime(df_users["first_seen_at"], errors="coerce")
df_events["event_time"] = pd.to_datetime(df_events["event_time"], errors="coerce")

In [5]:
# event_properties는 JSON 문자열로 저장되어 있으므로 딕셔너리로 변환
properties = df_events["event_properties"].apply(json.loads)

# 딕셔너리 안의 정보를 각각 컬럼으로 분리
properties_df = pd.DataFrame(properties.tolist())

# 기존 JSON 문자열 컬럼 대신 추출한 속성 컬럼 결합
df_events = pd.concat(
    [df_events.drop(columns=["event_properties"]), properties_df],
    axis=1,
)

# 이벤트에 따라 필요한 속성이 달라 빈칸이 생기는 것은 정상
print("추출된 속성:", properties_df.columns.tolist())

추출된 속성: ['total_questions', 'question_number', 'attempt_number', 'question_type', 'is_correct', 'progress_percent', 'energy_remaining', 'skip_reason']


In [6]:
# JSON에서 추출한 숫자 속성의 자료형 정리
number_columns = [
    "total_questions", "question_number", "attempt_number",
    "progress_percent", "energy_remaining",
]

for column in number_columns:
    df_events[column] = pd.to_numeric(df_events[column], errors="coerce")

# 정답(True) = 1, 오답(False) = 0, 답안 제출이 아닌 이벤트는 결측치 유지
df_events["is_correct"] = df_events["is_correct"].map({True: 1, False: 0})

# 사용자별로 이벤트를 시간순 정렬 (같은 시각이면 event_id 순서)
df_users = df_users.sort_values("user_id").reset_index(drop=True)
df_events = df_events.sort_values(
    ["user_id", "event_time", "event_id"]
).reset_index(drop=True)

## 데이터 구조 및 결측치 확인

In [7]:
# 각 CSV의 데이터 크기, 자료형, 결측치, 고유값 확인
for df_name, df in [("users", df_users), ("event_logs", df_events)]:
    print(f"\n[{df_name}] 행: {len(df):,}개 / 열: {len(df.columns)}개")

    summary_df = pd.DataFrame({
        "데이터타입": df.dtypes.astype(str),
        "결측치 개수": df.isna().sum(),
        "결측치 비율(%)": (df.isna().mean() * 100).round(2),
        "고유값 개수": df.nunique(),
    })
    display(summary_df)

# 전처리된 데이터 앞부분 확인
display(df_users.head())
display(df_events.head())


[users] 행: 10,000개 / 열: 6개


,데이터타입,결측치 개수,결측치 비율(%),고유값 개수
user_id,str,0,0.0,10000
first_seen_at,datetime64[us],0,0.0,9993
course_type,str,0,0.0,2
starting_point,str,0,0.0,2
self_assessed_level,int64,0,0.0,5
daily_goal_min,int64,0,0.0,4



[event_logs] 행: 320,705개 / 열: 14개


,데이터타입,결측치 개수,결측치 비율(%),고유값 개수
event_id,str,0,0.00,320705
user_id,str,0,0.00,10000
event_time,datetime64[us],0,0.00,314358
event_name,str,0,0.00,8
lesson_number,int64,0,0.00,2
lesson_attempt_id,str,3851,1.20,12815
total_questions,float64,307890,96.00,7
question_number,float64,30975,9.66,16
attempt_number,float64,171474,53.47,3
question_type,str,173937,54.24,4


,user_id,first_seen_at,course_type,starting_point,self_assessed_level,daily_goal_min
0,U00001,2026-08-13 03:34:38,beginner_english,start_basics,2,15
1,U00002,2026-08-31 12:55:17,intermediate_english,start_basics,3,15
2,U00003,2026-09-06 15:10:13,beginner_english,placement_test,3,10
3,U00004,2026-07-22 20:23:40,beginner_english,start_basics,3,15
4,U00005,2026-09-25 15:30:20,beginner_english,start_basics,5,10


,event_id,user_id,event_time,event_name,lesson_number,lesson_attempt_id,total_questions,question_number,attempt_number,question_type,is_correct,progress_percent,energy_remaining,skip_reason
0,E000152795,U00001,2026-08-13 03:53:55,lesson_started,1,A0000001,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E000152796,U00001,2026-08-13 03:54:02,question_viewed,1,A0000001,NaN,1.0,1.0,word_selection,NaN,NaN,NaN,NaN
2,E000152799,U00001,2026-08-13 03:54:27,answer_submitted,1,A0000001,NaN,1.0,NaN,NaN,1.0,6.67,NaN,NaN
3,E000152801,U00001,2026-08-13 03:54:31,question_viewed,1,A0000001,NaN,2.0,1.0,sentence_completion,NaN,NaN,NaN,NaN
4,E000215879,U00002,2026-08-31 13:09:52,lesson_started,1,A0000002,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 사용자 및 이벤트 기초 EDA

In [13]:
# 사용자 프로필의 주요 항목별 분포 확인
user_columns = ["course_type", "starting_point", "self_assessed_level", "daily_goal_min"]

for column in user_columns:
    print("\n[{column}]")
    display(df_users[column].value_counts().sort_index().to_frame("사용자 수"))


[{column}]


,사용자 수
course_type,
beginner_english,7017
intermediate_english,2983



[{column}]


,사용자 수
starting_point,
placement_test,2482
start_basics,7518



[{column}]


,사용자 수
self_assessed_level,
1,2530
2,3031
3,2505
4,1450
5,484



[{column}]


,사용자 수
daily_goal_min,
3,2459
10,4070
15,2486
30,985


In [9]:
# 이벤트 종류별 발생 건수 (사용자 수가 아닌 로그 건수)
print("[이벤트 종류별 발생 건수]")
display(df_events["event_name"].value_counts().to_frame("이벤트 건수"))

# 첫 레슨과 두 번째 레슨에서 기록된 이벤트 건수 확인
print("[레슨 번호별 이벤트 건수]")
display(df_events["lesson_number"].value_counts().sort_index().to_frame("이벤트 건수"))

[이벤트 종류별 발생 건수]


,이벤트 건수
event_name,
question_viewed,145261
answer_submitted,140499
lesson_started,12815
lesson_halfway_reached,7824
lesson_completed,6485
subscription_offer_viewed,3851
hint_used,2463
question_skipped,1507


[레슨 번호별 이벤트 건수]


,이벤트 건수
lesson_number,
1,317890
2,2815


## 기술통계 및 날짜 분포

숫자로 계산할 의미가 있는 항목만 `describe()`로 확인. 이벤트 ID, 레슨 번호 등은 기술통계에서 제외.

In [10]:
# 사용자 자기평가 수준(1~5)과 일일 목표 시간(분) 기술통계
print("[사용자 특성 기술통계]")
display(df_users[["self_assessed_level", "daily_goal_min"]].describe().T)

# 첫 레슨 시작 시 기록된 총 문제 수의 기술통계
first_starts = df_events[
    (df_events["event_name"] == "lesson_started") &
    (df_events["lesson_number"] == 1)
]
print("[첫 레슨 문제 수 기술통계]")
display(first_starts[["total_questions"]].describe().T)

# 첫 레슨 완료 시 기록된 에너지 잔량의 기술통계
first_completions = df_events[
    (df_events["event_name"] == "lesson_completed") &
    (df_events["lesson_number"] == 1)
]
print("[첫 레슨 완료 시 에너지 잔량 기술통계]")
display(first_completions[["energy_remaining"]].describe().T)

[사용자 특성 기술통계]


,count,mean,std,min,25%,50%,75%,max
self_assessed_level,10000.0,2.4327,1.154125,1.0,1.0,2.0,3.0,5.0
daily_goal_min,10000.0,11.4917,7.446105,3.0,10.0,10.0,15.0,30.0


[첫 레슨 문제 수 기술통계]


,count,mean,std,min,25%,50%,75%,max
total_questions,10000.0,14.431,1.360597,10.0,14.0,15.0,15.0,16.0


[첫 레슨 완료 시 에너지 잔량 기술통계]


,count,mean,std,min,25%,50%,75%,max
energy_remaining,6485.0,12.309792,7.507601,0.0,6.0,12.0,19.0,25.0


In [11]:
# 실제 생성된 사용자 유입 기간과 이벤트 발생 기간 확인
print("사용자 유입 기간:", df_users["first_seen_at"].min(), "~", df_users["first_seen_at"].max())
print("이벤트 발생 기간:", df_events["event_time"].min(), "~", df_events["event_time"].max())

# 신규 사용자 유입을 월 단위로 집계
monthly_users = df_users["first_seen_at"].dt.to_period("M").value_counts().sort_index()
display(monthly_users.to_frame("신규 사용자 수"))

사용자 유입 기간: 2026-07-01 00:42:39 ~ 2026-09-30 23:34:54
이벤트 발생 기간: 2026-07-01 01:09:07 ~ 2026-10-06 19:50:34


,신규 사용자 수
first_seen_at,
2026-07,3429
2026-08,3346
2026-09,3225


## 최종 품질 확인 및 파일 저장

중복, 날짜, 사용자 ID 연결 여부를 마지막에 확인하고 `data/processed/`에 저장.

> `subscription_offer_viewed`의 `lesson_attempt_id`는 원래 비어 있으므로 삭제하지 않음.

In [12]:
# 사용자 및 이벤트 고유 ID 중복 확인
user_duplicates = df_users["user_id"].duplicated().sum()
event_duplicates = df_events["event_id"].duplicated().sum()

# users.csv에 없는 사용자의 이벤트 기록 확인
missing_users = (~df_events["user_id"].isin(df_users["user_id"])).sum()

# 날짜 변환 실패 여부 확인
missing_dates = df_users["first_seen_at"].isna().sum() + df_events["event_time"].isna().sum()

print("사용자 ID 중복:", user_duplicates)
print("이벤트 ID 중복:", event_duplicates)
print("등록되지 않은 사용자 이벤트:", missing_users)
print("날짜 변환 실패:", missing_dates)

# 정상적인 시도 ID 결측 이벤트 확인
display(df_events.loc[
    df_events["lesson_attempt_id"].isna(), "event_name"
].value_counts().to_frame("lesson_attempt_id 결측 건수"))

# 문제가 있으면 저장 중단, 이상이 없으면 처리된 CSV 저장
if user_duplicates + event_duplicates + missing_users + missing_dates > 0:
    raise ValueError("데이터 품질에 문제가 있으므로 저장 전 확인 필요")

USER_PROCESSED_PATH = DATA_PROCESSED_DIR / "users_preprocessed.csv"
EVENT_PROCESSED_PATH = DATA_PROCESSED_DIR / "event_logs_preprocessed.csv"

df_users.to_csv(USER_PROCESSED_PATH, index=False, encoding="utf-8")
df_events.to_csv(EVENT_PROCESSED_PATH, index=False, encoding="utf-8")

print("사용자 데이터 저장:", USER_PROCESSED_PATH)
print("이벤트 데이터 저장:", EVENT_PROCESSED_PATH)

사용자 ID 중복: 0
이벤트 ID 중복: 0
등록되지 않은 사용자 이벤트: 0
날짜 변환 실패: 0


,lesson_attempt_id 결측 건수
event_name,
subscription_offer_viewed,3851


사용자 데이터 저장: C:\Users\joon5\Documents\github\duolingo-funnel-analysis\data\processed\users_preprocessed.csv
이벤트 데이터 저장: C:\Users\joon5\Documents\github\duolingo-funnel-analysis\data\processed\event_logs_preprocessed.csv
